# A Hidden-State Failure

## When rerunning a perfectly ordinary cell silently changes the analysis

This demonstration notebook contains a very common interactive-programming mistake:

> **A transformation modifies a DataFrame in place, so rerunning the same cell applies the transformation twice.**

The notebook runs correctly from top to bottom after a fresh kernel restart.

But if an analyst reruns one ordinary-looking transformation cell during exploration, the results silently become wrong.

# Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cell 2 — Load external data

In [ ]:
rng = np.random.default_rng(556)

n = 120

df = pd.DataFrame({
    "site": rng.choice(
        ["North", "Central", "South"],
        size=n
    ),
    "sample_size": rng.integers(
        80,
        250,
        size=n
    ),
    # External source reports rates as percentages.
    "success_rate": np.clip(
        rng.normal(
            loc=82,
            scale=7,
            size=n
        ),
        55,
        99
    )
})

df.head()

### Sanity check

At ingestion, `success_rate` is expressed as a **percentage**.

Values around 80 therefore mean approximately 80%, not 80 as a proportion.

In [ ]:
df["success_rate"].describe()

# Cell 3 — Convert percent to proportion

This is the problematic cell.

It looks completely ordinary:

In [ ]:
df["success_rate"] = (
    df["success_rate"] / 100
)

df["success_rate"].head()

# Cell 4 — Analysis

In [ ]:
summary = (
    df.groupby("site")
      .agg(
          n_sites=("success_rate", "size"),
          mean_success_rate=("success_rate", "mean"),
          sd_success_rate=("success_rate", "std")
      )
)

summary

# Cell 5 — Overall reported result

In [ ]:
overall_mean = df["success_rate"].mean()

print(
    f"Overall mean success rate: "
    f"{overall_mean:.3f}"
)

# Cell 6 — Figure

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for site, data in df.groupby("site"):
    ax.scatter(
        data["sample_size"],
        data["success_rate"],
        label=site,
        alpha=0.7
    )

ax.set(
    title="Success rate by sample size",
    xlabel="Sample size",
    ylabel="Success rate (proportion)"
)

ax.legend(title="Site")
fig.tight_layout()

plt.show()